# Brand text embeddings

This notebook starts from the curated Kaggle dataset and turns the brand vibe text into normalised vectors for clustering and recommendations.


## Setup

In [ ]:
from pathlib import Path

CWD = Path.cwd().resolve()
ROOT = CWD.parent if CWD.name == "step_2_text_embeddings" else CWD
STEP_DIR = ROOT / "step_2_text_embeddings"

helper_path = STEP_DIR / "helpers" / "embedding_helpers.py"
exec(compile(helper_path.read_text(encoding="utf-8"), str(helper_path), "exec"), globals())

source_path = ROOT / "helpers" / "source_helpers.py"
exec(compile(source_path.read_text(encoding="utf-8"), str(source_path), "exec"), globals())

print("Root:", ROOT)
print("Embedding output:", EMBEDDINGS_PATH)


## Data

Step 2 uses the cleaned category CSVs from Kaggle. In a clean checkout, `DOWNLOAD_DATASET_FROM_KAGGLE = True` fills `final_dataset/` before embedding.


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

DOWNLOAD_DATASET_FROM_KAGGLE = True
REFRESH_KAGGLE_DATASET = False
VIBE_COLS = ["aesthetic_keywords", "silhouettes", "materials", "palette"]

dataset_dir = ensure_final_dataset(
    ROOT,
    allow_download=DOWNLOAD_DATASET_FROM_KAGGLE,
    force_download=REFRESH_KAGGLE_DATASET,
)

brands_all = align_to_thesis_dataset(load_final_dataset(ROOT, drop_empty_vibe_rows=False))
brands = align_to_thesis_dataset(load_final_dataset(ROOT, drop_empty_vibe_rows=True))
counts = pd.DataFrame([
    {"category": CATEGORY_LABELS[category], "rows": int(brands_all['category'].eq(category).sum())}
    for category in CATEGORIES
])

print(f"Kaggle dataset: {KAGGLE_DATASET}")
print(f"Dataset folder: {dataset_dir}")
print(f"Loaded {len(brands_all):,} curated rows; {len(brands):,} after dropping all-empty vibe rows.")
print()
print("Brands per category:")
display(counts)
save_table(counts, "dataset_category_counts", ROOT)

brands.head(3)


## Text

One short prompt per brand from the vibe fields.


In [ ]:
brands["prompt"] = brands.apply(to_prompt, axis=1)

# Prompt samples.
for _, row in brands.sample(3, random_state=RNG_SEED).iterrows():
    print(f"[{row['category']}] {row['brand_name']}")
    print(f"  {row['prompt']}")
    print()


In [ ]:
# Prompt length summary.
word_counts = brands["prompt"].str.split().str.len()
prompt_summary = pd.DataFrame({
    "metric": ["min words", "median words", "max words", "empty prompts"],
    "value": [int(word_counts.min()), int(word_counts.median()), int(word_counts.max()), int((word_counts == 0).sum())],
})
display(prompt_summary)
save_table(prompt_summary, "embedding_prompt_length_summary", ROOT)


## Embeddings

Sentence-transformer vectors, normalised for cosine similarity.


In [ ]:
from sentence_transformers import SentenceTransformer
import time

print(f"Loading model: {EMBEDDING_MODEL}")
model = SentenceTransformer(EMBEDDING_MODEL)

prompts = brands["prompt"].tolist()
BATCH_SIZE = 64
n_batches = (len(prompts) + BATCH_SIZE - 1) // BATCH_SIZE

print()
print(f"Brands to encode : {len(prompts)}")
for category in CATEGORIES:
    print(f"  {category:<9}: {(brands['category'] == category).sum()}")
print(f"Batch size       : {BATCH_SIZE}  ({n_batches} batches)")
print()

t0 = time.time()
all_batches = []

for i, start in enumerate(range(0, len(prompts), BATCH_SIZE)):
    batch = prompts[start : start + BATCH_SIZE]
    vecs = model.encode(batch, convert_to_numpy=True, show_progress_bar=False)
    all_batches.append(vecs)

    done = i + 1
    if done == 1 or done == n_batches or done % 5 == 0:
        elapsed_so_far = time.time() - t0
        eta = (elapsed_so_far / done) * (n_batches - done)
        print(f"Batch {done}/{n_batches} done. ETA {eta:.0f}s")

raw_embeddings = np.vstack(all_batches)
elapsed = time.time() - t0

# Cosine-similarity normalisation.
norms = np.linalg.norm(raw_embeddings, axis=1, keepdims=True)
norms[norms == 0.0] = 1.0
embeddings = raw_embeddings / norms

print(f"Embeddings shape : {embeddings.shape}")
print(f"Embedding dim    : {embeddings.shape[1]}")
print(f"Time elapsed     : {elapsed:.1f} s  ({elapsed / len(prompts) * 1000:.1f} ms/brand)")


## Files

Embeddings and matching metadata.


In [ ]:
# Embedding archive.
np.savez_compressed(
    EMBEDDINGS_PATH,
    embeddings=embeddings.astype(np.float32),
    model_name=np.array([EMBEDDING_MODEL]),
)
print(f"Saved embeddings -> {EMBEDDINGS_PATH}  ({EMBEDDINGS_PATH.stat().st_size / 1024:.0f} KB)")

# Metadata table.
save_cols = ["brand_name", "category", "official_website"] + VIBE_COLS + ["prompt", "male", "female", "children", "price"]
brands[[c for c in save_cols if c in brands.columns]].to_csv(METADATA_PATH, index=False)
print(f"Saved metadata   -> {METADATA_PATH}  ({METADATA_PATH.stat().st_size / 1024:.0f} KB)")


## Neighbour check

Nearest-brand check after reload.


In [ ]:
# Reloaded files.
archive = np.load(EMBEDDINGS_PATH, allow_pickle=True)
emb_loaded  = archive["embeddings"]
model_saved = archive["model_name"][0]
meta_loaded = pd.read_csv(METADATA_PATH)

assert emb_loaded.shape[0] == len(meta_loaded), "row count mismatch!"
print(f"Reloaded {emb_loaded.shape[0]} brands x {emb_loaded.shape[1]} dims")
print(f"Model recorded in file: {model_saved}")


In [ ]:
# Example neighbours.
probes = [
    ("Toteme", "clothes"),
    ("Gianvito Rossi", "shoes"),
    ("Bottega Veneta", "bags"),
]

for probe, category in probes:
    if ((meta_loaded["brand_name"] == probe) & (meta_loaded["category"] == category)).any():
        print()
        print(f"--- Nearest neighbours for '{probe}' ({category}) ---")
        display(nearest_neighbours(probe, category=category, k=5))
    else:
        print()
        print(f"'{probe}' ({category}) not found - skipping.")


## Files for later notebooks

Files used by the later notebooks.


In [ ]:
metadata_check, embedding_check, resolved_embedding_dir = load_embeddings(ROOT)
embedding_output_summary = pd.DataFrame([
    {"artifact": "brand_embeddings.npz", "path": str(resolved_embedding_dir / "brand_embeddings.npz"), "rows": embedding_check.shape[0], "columns": embedding_check.shape[1]},
    {"artifact": "brand_metadata.csv", "path": str(resolved_embedding_dir / "brand_metadata.csv"), "rows": len(metadata_check), "columns": len(metadata_check.columns)},
])
display(embedding_output_summary)
save_table(embedding_output_summary, "embedding_output_summary", ROOT)
